# LeetCode Bot - Usage Analytics

Analyses daily CSV logs from the Telegram bot's analytics system.
Each CSV contains: `timestamp, user_id, username, first_name, last_name, language_code, chat_id, chat_type, command, arguments`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os

csv_files = sorted(glob.glob(os.path.join(os.path.dirname(os.path.abspath("__file__")), "*.csv")))
if not csv_files:
    raise FileNotFoundError("No analytics CSV files found. Run the bot and send some commands first.")

df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()

print(f"Loaded {len(df):,} events from {len(csv_files)} file(s), spanning {df['date'].nunique()} day(s)")
df.head(10)

: 

## Active Users Per Day

In [ ]:
daily_users = df.groupby("date")["user_id"].nunique()
ax = daily_users.plot(kind="bar", figsize=(12, 4), color="steelblue")
ax.set_title("Unique Active Users Per Day")
ax.set_ylabel("Users")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Command Popularity

In [ ]:
cmd_counts = df["command"].value_counts()
ax = cmd_counts.plot(kind="barh", figsize=(10, 6), color="coral")
ax.set_title("Command Popularity")
ax.set_xlabel("Count")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Usage by Hour of Day (UTC)

In [ ]:
hourly = df.groupby("hour")["user_id"].count()
ax = hourly.plot(kind="bar", figsize=(12, 4), color="mediumseagreen")
ax.set_title("Events by Hour of Day (UTC)")
ax.set_xlabel("Hour")
ax.set_ylabel("Events")
plt.tight_layout()
plt.show()

## Language Code Distribution (Proxy for Locale)

In [ ]:
lang = df[df["language_code"] != ""]["language_code"].value_counts().head(15)
if not lang.empty:
    ax = lang.plot(kind="pie", autopct="%1.1f%%", figsize=(8, 8))
    ax.set_title("Language Code Distribution")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()
else:
    print("No language_code data available yet.")

## Chat Type Distribution

In [ ]:
chat_types = df["chat_type"].value_counts()
ax = chat_types.plot(kind="pie", autopct="%1.1f%%", figsize=(6, 6))
ax.set_title("Chat Type Distribution (Private vs Group)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## User Retention (Returning Users Over Time)

In [ ]:
user_first_seen = df.groupby("user_id")["date"].min().rename("first_seen")
user_sessions = df.groupby(["user_id", "date"]).size().reset_index(name="events")
user_sessions = user_sessions.merge(user_first_seen, on="user_id")
user_sessions["is_returning"] = user_sessions["date"] > user_sessions["first_seen"]

retention = user_sessions.groupby("date").agg(
    total_users=("user_id", "nunique"),
    returning_users=("is_returning", "sum"),
)
retention["returning_pct"] = (retention["returning_users"] / retention["total_users"] * 100).round(1)

ax = retention["returning_pct"].plot(kind="line", marker="o", figsize=(12, 4), color="purple")
ax.set_title("Returning User Rate Over Time")
ax.set_ylabel("% Returning Users")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

retention

## Most Viewed Problems

In [ ]:
problem_cmds = df[df["command"].isin(["problem", "solution"])]
if not problem_cmds.empty:
    top = problem_cmds["arguments"].value_counts().head(20)
    ax = top.plot(kind="barh", figsize=(10, 6), color="goldenrod")
    ax.set_title("Most Viewed Problems / Solutions")
    ax.set_xlabel("Views")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No /problem or /solution commands recorded yet.")

## Per-User Activity Summary

In [ ]:
user_summary = df.groupby(["user_id", "username"]).agg(
    total_events=("command", "count"),
    unique_commands=("command", "nunique"),
    first_seen=("date", "min"),
    last_seen=("date", "max"),
    days_active=("date", "nunique"),
    language=("language_code", "first"),
).sort_values("total_events", ascending=False)

user_summary